<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will understand the core idea of <b>DPO rewriting</b> using small toy graphs: match <b>L</b>, preserve <b>K</b>, produce <b>R</b>.
</div>

# S03 · DPO basics on toy graphs (L ← K → R)

This notebook is graph-only (no chemistry). It builds intuition before we touch molecules.


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: L/K/R and rewrite intuition
- Hands-on: implement a tiny rewrite and inspect results


# Theory

A DPO rule is often written as:
- L (left pattern)
- K (interface/shared part)
- R (right replacement)

Application (intuition):
1) Find L in a host graph
2) Delete L\K
3) Glue in R\K along K

Chemistry needs extra guards (valence, charge, sanitization) — later notebooks cover that.


# Practical


In [ ]:
import networkx as nx
from dataclasses import dataclass
from networkx.algorithms import isomorphism as iso

@dataclass(frozen=True)
class Rule:
    L: nx.Graph
    K: nx.Graph
    R: nx.Graph
    name: str = "rule"

def apply_dpo_once(G: nx.Graph, rule: Rule) -> list[nx.Graph]:
    nm = iso.categorical_node_match("label", None)
    em = iso.categorical_edge_match("label", None)
    GM = iso.GraphMatcher(G, rule.L, node_match=nm, edge_match=em)

    outs = []
    for m in GM.subgraph_isomorphisms_iter():
        H = G.copy()
        L_nodes = set(rule.L.nodes())
        K_nodes = set(rule.K.nodes())

        # delete nodes in L\K
        for ln in (L_nodes - K_nodes):
            H.remove_node(m[ln])

        # glue in R\K with fresh node ids
        next_id = max([n for n in H.nodes if isinstance(n, int)], default=-1) + 1
        fresh = {}
        for rn, d in rule.R.nodes(data=True):
            if rn in K_nodes:
                continue
            fresh[rn] = next_id
            H.add_node(next_id, **d)
            next_id += 1

        label_to_host = {rule.L.nodes[k]["label"]: m[k] for k in K_nodes}

        def host_id(rn):
            if rn in K_nodes:
                lbl = rule.R.nodes[rn]["label"]
                return label_to_host[lbl]
            return fresh[rn]

        for u, v, d in rule.R.edges(data=True):
            hu, hv = host_id(u), host_id(v)
            if not H.has_edge(hu, hv):
                H.add_edge(hu, hv, **d)

        outs.append(H)
    return outs

# Host graph: C--Cl
G = nx.Graph()
G.add_node(0, label="C")
G.add_node(1, label="Cl")
G.add_edge(0, 1, label=1)

# Rule: replace Cl with O (keep C as K)
L = nx.Graph(); L.add_node("a", label="C"); L.add_node("b", label="Cl"); L.add_edge("a","b", label=1)
K = nx.Graph(); K.add_node("a", label="C")
R = nx.Graph(); R.add_node("a", label="C"); R.add_node("c", label="O"); R.add_edge("a","c", label=1)

rule = Rule(L=L, K=K, R=R, name="Cl_to_O")
outs = apply_dpo_once(G, rule)
print("Applications:", len(outs))
print("Before:", list(G.nodes(data=True)))
print("After:", list(outs[0].nodes(data=True)) if outs else None)


# Discussion
- The toy rewrite ignores many real constraints. That's intentional: first learn the mechanism.
- In chemical rewriting, the critical step is ensuring the modified structure remains chemically valid.


# Quiz
1. What information should K contain?
2. Why can a rule apply multiple times to one host graph?
3. What is the 'dangling edge' issue in DPO terms?


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
